# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all record sets, their @id and fields' @id
def list_record_sets(dataset):
    # Access record sets via metadata
    if not hasattr(dataset.metadata, "record_sets"):
        print("No record sets are defined in the dataset metadata.")
        return
    record_sets = dataset.metadata.record_sets
    if not record_sets:
        print("No record sets are present in the dataset metadata.")
        return
    for rs in record_sets:
        print(f"Record set: {rs.name}")
        print(f"  @id: {rs.id}")
        print(f"  Fields:")
        for field in rs.fields:
            print(f"    - {field.name}  (@id: {field.id})  [Type: {getattr(field, 'data_type', 'N/A')}]" )
        print("")

list_record_sets(dataset)

## 3. Data Extraction
Load data from record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview above.

> 💡 **Note**: For demonstration, we extract the main record set (with tabular/clinical data) if present.

In [ ]:
# Find the relevant record sets @id
record_sets = []
record_set_id_to_name = {}
if hasattr(dataset.metadata, "record_sets"):
    for rs in dataset.metadata.record_sets:
        record_sets.append(rs.id)
        record_set_id_to_name[rs.id] = rs.name

if not record_sets:
    print("No record sets available!")
else:
    print(f"Record set @ids found: {record_sets}")

# Extract data from each record set
dataframes = {}
for record_set in record_sets:
    print(f"Loading records from: {record_set} ({record_set_id_to_name[record_set]})")
    records = list(dataset.records(record_set=record_set))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set] = df
        print(f"Loaded DataFrame with shape: {df.shape}")
    else:
        print(f"No records found for record set {record_set}")

# For demonstration, pick the first non-empty record set (usually primary table)
main_rs_id = None
for rsid, df in dataframes.items():
    if not df.empty:
        main_rs_id = rsid
        break

if main_rs_id:
    print(f"\nMain record set selected for EDA: {main_rs_id} ({record_set_id_to_name.get(main_rs_id, main_rs_id)})")
    print("Columns/fields (@id):", list(dataframes[main_rs_id].columns))
    display(dataframes[main_rs_id].head())
else:
    print("No usable dataframes could be loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

> **Note:** The specific field `@id`s for numeric and grouping fields should be chosen based on actual dataset columns. Here, we demonstrate with the first detected numeric field.

In [ ]:
import numpy as np

if not main_rs_id:
    print("No main record set available for EDA.")
else:
    df = dataframes[main_rs_id]
    numeric_field_id = None
    group_field_id = None

    # Try to pick a numeric field using pandas dtypes and column names
    for col in df.columns:
        if np.issubdtype(df[col].dropna().infer_objects().dtype, np.number):
            numeric_field_id = col
            break

    # Pick a groupable (categorical) field
    for col in df.columns:
        if col != numeric_field_id and (df[col].dtype == object or pd.api.types.is_categorical_dtype(df[col])):
            group_field_id = col
            break

    if numeric_field_id is None:
        print("No numeric field found for demonstration.")
    else:
        threshold = df[numeric_field_id].dropna().mean()  # For demonstration, use the mean as threshold
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by categorical field if one found
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped data by {group_field_id} (showing mean {numeric_field_id}):")
            display(grouped_df)
        else:
            print("No suitable grouping field found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Example: plot a histogram for a numeric field and a bar chart for group means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not main_rs_id or numeric_field_id is None:
    print("No numeric data available for visualization.")
else:
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    # Histogram of numeric field in full main df
    sns.histplot(df[numeric_field_id].dropna(), bins=10, ax=ax[0], kde=True)
    ax[0].set_title(f"Distribution of {numeric_field_id}")
    ax[0].set_xlabel(numeric_field_id)

    if group_field_id:
        grouped_bar = df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        sns.barplot(data=grouped_bar, x=group_field_id, y=numeric_field_id, ax=ax[1])
        ax[1].set_title(f"Mean {numeric_field_id} by {group_field_id}")
        ax[1].set_xticklabels(ax[1].get_xticklabels(), rotation=45, ha="right")
    else:
        ax[1].axis('off')

    plt.tight_layout()
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

<br>
* In this notebook, we loaded and explored the FAIR² tabular dataset on second primary colorectal cancer in cancer survivors using the `mlcroissant` library, referencing all entities by their `@id`.
* The data were successfully loaded into DataFrames, with fields and record sets identified by `@id`.
* Basic statistical filtering, normalization, grouping, and visualization demonstrated the ease of analysis using Croissant datasets in Python workflows.
* Further analyses should be based on domain-relevant questions and clinical/phenotypic interpretation.
